In [ ]:
list_undertrained = []
with open('./Qwen_Qwen2_5_7B.md','r',encoding='utf-8') as f:
    for line in f:
        splits = line.strip().split('|')
        if len(splits) == 7:
            try:
                list_undertrained.append(int(splits[1].strip()))
            except: continue
            
len(list_undertrained)

In [ ]:
list_unreachable_tokens = []
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B")

for token_id in range(257,tokenizer.eos_token_id):
    re_tokenized_string = tokenizer.encode(tokenizer.decode(token_id))
    if re_tokenized_string[0] != token_id:
        list_unreachable_tokens.append(token_id)

len(list_unreachable_tokens)

In [ ]:
final_list_to_replace = list_undertrained
for token_id in list_unreachable_tokens:
    if token_id not in final_list_to_replace:
        final_list_to_replace.append(token_id)
len(final_list_to_replace)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B")
tokenizer.save_pretrained("./Qwen2.5-7B-Base/")

import json
tokenizer_json = json.load(open("./Qwen2.5-7B-Base/tokenizer.json",encoding='utf-8'))
merge_list = tokenizer_json['model']['merges']


import networkx as nx
G = nx.DiGraph()
for item in merge_list:
    item_token = ''.join(item)
    G.add_edge(item[0],item_token)
    G.add_edge(item[1],item_token)


In [ ]:
final_tokens_to_replace = [tokenizer.convert_ids_to_tokens(token_id) for token_id in final_list_to_replace]


def is_accessible_nodes(graph, start_node):
    if len(list(nx.descendants(graph, start_node))) > 0 :
        return True
    
    return False

final_tokens_can_replace = []
for node in final_tokens_to_replace:
    try:
        if not is_accessible_nodes(G,node):
            final_tokens_can_replace.append(tokenizer.convert_tokens_to_ids(node))
            print(node)
    except: pass

with open('./TokenIDs_Can_Replace_Qwen2.5-Magikarp+Unreachable.txt','w',encoding='utf-8') as f:
    for token_id in final_tokens_can_replace:
        f.write(str(token_id) + '\n')
f.close()



In [ ]:
from transformers import Qwen2Tokenizer, AutoTokenizer

tokenizer = Qwen2Tokenizer.from_pretrained("Qwen/Qwen2.5-7B", use_fast=False)
tokenizer.save_pretrained("./Qwen2.5-7B-Base")

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B")
tokenizer.save_pretrained("./Qwen2.5-7B-Base/")


del tokenizer

In [ ]:
import json
vocab_json = json.load(open("./Qwen2.5-7B-BASE/vocab.json",encoding='utf-8'))
merges_txt = open("./Qwen2.5-7B-BASE/merges.txt",encoding='utf-8').readlines()[1:]

In [ ]:
tokenids_to_replace = open('./TokenIDs_Can_Replace_Qwen2.5-Magikarp+Unreachable.txt').read().splitlines()
tokenids_to_replace = [int(x.strip()) for x in tokenids_to_replace]

In [ ]:
from transformers import Qwen2Tokenizer
base_tokenizer = Qwen2Tokenizer.from_pretrained('Qwen/Qwen2.5-7B', use_fast=False)
tokens_to_replace = [base_tokenizer.convert_ids_to_tokens(x) for x in tokenids_to_replace]
len(tokens_to_replace)

In [ ]:
tokens_to_add = []
for line in open('./VocabFiles/CPT-SupremeCourtCase/CPT-SupremeCourtCase-Qwen2.5_Vocab/10.0_.txt','r',encoding='utf-8'):
    tokens_to_add.append(line.strip())

In [ ]:
tokens_to_add = sorted(tokens_to_add, key=lambda x: len(x))

In [ ]:
tokens_dump = {}
tokens_to_add = sorted(tokens_to_add, key=lambda x: len(x))

def checkList(list_tokens, vocab_to_check):
    # print('++Checking list of tokens:', list_tokens)
    merged_tokens = list_tokens[:]
    longest_token = ""
    start_idx, end_idx = -1, -1
    
    for i in range(len(list_tokens)):
        for j in range(i + 1, len(list_tokens) + 1):
            candidate = "".join(list_tokens[i:j])
            if candidate in vocab_to_check:
                if len(candidate) > len(longest_token):
                    longest_token = candidate
                    start_idx, end_idx = i, j

    if longest_token:
        merged_tokens = list_tokens[:start_idx] + [longest_token] + list_tokens[end_idx:]

    # print('--Merged tokens:', merged_tokens)
    return merged_tokens

vocab_base = base_tokenizer.get_vocab()


for k in tokens_to_add:
    drop_words = []
    for idx,word in enumerate(tokens_to_add):
        if word in vocab_base:
            drop_words.append(idx)
    
    for idx in drop_words[::-1]: del tokens_to_add[idx]
    
    # if 'Ĕ' in k: continue
    # if 'Ä' in k: continue
    # if 'ą' in k: continue
        
    split = base_tokenizer.tokenize(k if not k.startswith('Ġ') else ' '+k[1:])
    
    # flag = False
    # for sp in split:
    #     if 'Ä' in sp: 
    #         flag = True
    #         break
        
    # if flag: continue

    
    if len(split) == 1:
        continue
    
    elif len(split) == 2:
        tokens_dump[k] = [split[0],split[1]]
    
    else:
        new_split = checkList(split, tokens_dump)
        # print('new_split:',new_split)
        if len(new_split) == 2:
            tokens_dump[k] = [new_split[0],new_split[1]]
            continue
        
        new_word=new_split[0]
        for i in range(1,len(new_split)):
            left = new_word
            right = new_split[i]
            new_word += new_split[i]
            # print('new_word:',new_word, 'Merge:',left,right)
            if new_word not in tokens_dump:
                tokens_dump[new_word] = [left,right]

len(tokens_dump)

In [ ]:
final_tokens_to_add = []
for key,val in tokens_dump.items():
    if len(final_tokens_to_add) == len(tokenids_to_replace): break
    
    final_tokens_to_add.append([key]+val)


len(final_tokens_to_add)

In [ ]:
final_tokens_to_add[-1]

In [ ]:
vocab_json = json.load(open("./Qwen2.5-7B-BASE/vocab.json",encoding='utf-8'))
vocab_json


In [ ]:
final_tokenid_to_replace = tokenids_to_replace[:len(final_tokens_to_add)]
len(final_tokenid_to_replace)


In [ ]:
vocab_json_dump = {}

for k,v in vocab_json.items():
    if v not in final_tokenid_to_replace:
        vocab_json_dump[k] = v

In [ ]:
len(vocab_json), len(vocab_json_dump)

In [ ]:
tokens_replaced = []
idx=0
for k,v in zip(final_tokens_to_add,final_tokenid_to_replace):
    print(k[0],v)
    vocab_json_dump[k[0]] = v
    
    tokens_replaced.append([v,tokens_to_replace[idx],k[0],base_tokenizer.encode(k[0].replace('Ġ',' '))])
    idx +=1

In [ ]:
for key,val in tokens_dump.items():
    print(key,val)

In [ ]:
for key,val in tokens_dump.items():
    if key not in vocab_json_dump:
        print('Adding:',key)
        vocab_json_dump[key] = len(vocab_json_dump)


In [ ]:
tokens_replaced

In [ ]:
dump_dir = './VocabFiles/CPT-SupremeCourtCase/CPT-SupremeCourtCase-Qwen2.5_Tokenizers/SupremeCourtCase_DomainSpecific_MEDVOC_Qwen2.5_10.0_Replaced_MagikarpUnreachable/'
base_tokenizer.save_pretrained(dump_dir)

import pickle as pkl
with open(f'{dump_dir}/replaced_tokens.pkl','wb') as f:
    pkl.dump(tokens_replaced,f)
f.close()

In [ ]:
with open(f'{dump_dir}/vocab.json','w',encoding='utf-8') as f:
    json.dump(vocab_json_dump,f)
f.close()

In [ ]:
merges_txt_dump = []
for merge in merges_txt:
    if ''.join(merge.strip().split()) in [x[1] for x in tokens_replaced]:
        continue
    else:
        merges_txt_dump.append(merge)

len(merges_txt), len(merges_txt_dump)

In [ ]:
for item in tokens_dump.items():
    print(item)
    assert len(item) == 2
    
    key,left,right = item[0], item[1][0], item[1][1]
    
    merges_txt_dump.append(f"{left} {right}\n")

In [ ]:
merges_txt

In [ ]:
with open(f'{dump_dir}/merges.txt','w',encoding='utf-8') as f:
    f.write(f"#version: 0.2\n")
    f.write(''.join(merges_txt_dump))

In [1]:
from transformers import AutoTokenizer
dump_dir = './VocabFiles/CPT-SupremeCourtCase/CPT-SupremeCourtCase-Qwen2.5_Tokenizers/SupremeCourtCase_DomainSpecific_MEDVOC_Qwen2.5_10.0_Replaced_MagikarpUnreachable/'
modi_tokenizer = AutoTokenizer.from_pretrained(f"{dump_dir}")


In [ ]:
modi_tokenizer.save_pretrained(f"{dump_dir}")


In [ ]:
import pandas as pd
from transformers import AutoTokenizer
from tqdm import tqdm

df_MCQ = pd.read_csv('../ICLR-Test-dataset/SplitMoreThan1-Qwen2.5.csv')
df_CHQ = pd.read_csv('../TGT-Data/CHQ/SplitMoreThan1-Qwen2.5.csv')
df_OPI = pd.read_csv('../TGT-Data/OPI-RRS/SplitMoreThan1-Qwen2.5.csv')
df = pd.concat([df_MCQ, df_CHQ, df_OPI], axis=0)

freq_ebm = df['Count'].to_list()
terms_EBM = df['Word'].to_list()
split_bart = df['Splits'].to_list()


domain_tok = AutoTokenizer.from_pretrained('./VocabFiles/CPT/CPT_Vocab_Qwen2.5_EMNLP/CPT_CPT_8.0_/')
domain_tok_rep = AutoTokenizer.from_pretrained('./VocabFiles/CPT/CPT_8.0_Vocab_Qwen2.5_EMNLP_Replaceed/')

for idx,term in enumerate(terms_EBM):
    split_token = domain_tok.tokenize(term)
    split_token_rep = domain_tok_rep.tokenize(term)
    
    if len(split_token) < len(split_token_rep):
        print('--Split More:',term,split_token,split_token_rep)
    elif len(split_token) > len(split_token_rep):
        print('++Split Less:',term,split_token,split_token_rep)
    else: continue
        




